In [80]:
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment

def permutate_al_atoms(frac_coords: torch.Tensor, atom_types: torch.Tensor, sigma: float) -> torch.Tensor:
    """
    Permute aluminum atoms in the unit cell by adding noise and assigning them to the closest
    positions in the original coordinates, while swapping types to maintain the composition.

    Args:
        frac_coords (torch.Tensor): Fractional coordinates of the unit cell.
        atom_types (torch.Tensor): Atom types of the unit cell.
        sigma (float): Standard deviation of the noise to add to the aluminum atoms' positions.

    Returns:
        torch.Tensor: Permutated atom types.
        torch.Tensor: Noisy coordinates of aluminum atoms.
    """
    # Get the indices of aluminum atoms (0 indicates aluminum)
    al_indices = torch.where(atom_types == 0)[0]

    # Create noisy positions for aluminum atoms
    noisy_coords = frac_coords.clone()
    noise = torch.normal(mean=0.0, std=sigma, size=(len(al_indices), 3)).to(frac_coords.device)
    noisy_coords[al_indices] += noise

    # Apply periodic boundary conditions (wrap coordinates to [0, 1))
    noisy_coords %= 1.0

    # Extract noisy coordinates for aluminum atoms only
    noisy_al_coords = noisy_coords[al_indices]

    # Convert coordinates to numpy for linear_sum_assignment
    coords_np = frac_coords.detach().cpu().numpy()
    noisy_al_coords_np = noisy_al_coords.detach().cpu().numpy()

    # Calculate distance matrix between noisy aluminum coordinates and all original coordinates
    dist_matrix = np.linalg.norm(
        np.minimum(np.abs(coords_np[:, None, :] - noisy_al_coords_np[None, :, :]),
                   1 - np.abs(coords_np[:, None, :] - noisy_al_coords_np[None, :, :])),
        axis=-1
    )

    # Solve the optimal transport (linear assignment) problem
    row_ind, col_ind = linear_sum_assignment(dist_matrix)
    print("Al indices", al_indices)
    print("Row ind", row_ind)
    print("Col ind", col_ind)

    # Create a copy of atom types to apply the permutation
    permuted_types = atom_types.clone()

    # Swap the atom types between aluminum atoms and the atoms they replace
    for orig_idx, new_idx in zip(row_ind, col_ind):
        # Swap types: set the type at `new_idx` to aluminum (0) and the type at `orig_idx` to what was at `new_idx`
        permuted_types[al_indices[new_idx]] = atom_types[orig_idx]
        permuted_types[orig_idx] = atom_types[al_indices[new_idx]]

    return permuted_types, noisy_coords, noise

In [81]:
# Step 1: Generate 20 random fractional coordinates (values between 0 and 1)
num_atoms = 20
coords = torch.rand((num_atoms, 3))  # Shape [20, 3]

# Step 2: Generate atom type assignments with an approximately 85% Si and 15% Al ratio
# Calculate number of Si (1) and Al (0)
num_si = int(num_atoms * 0.85)
num_al = num_atoms - num_si

# Create the atom type array
atom_types = torch.tensor([1] * num_si + [0] * num_al)

# Shuffle the atom types to randomize their order
atom_types = atom_types[torch.randperm(num_atoms)]

noise = 0.3

In [82]:
print(coords)
print(atom_types)


tensor([[0.0165, 0.5861, 0.2502],
        [0.5998, 0.9031, 0.8464],
        [0.9129, 0.8250, 0.1391],
        [0.9210, 0.5640, 0.1489],
        [0.9313, 0.2000, 0.6341],
        [0.4447, 0.7558, 0.9236],
        [0.2581, 0.8302, 0.6531],
        [0.4812, 0.0150, 0.7531],
        [0.0948, 0.9301, 0.9043],
        [0.4719, 0.4274, 0.4080],
        [0.2282, 0.6136, 0.9391],
        [0.9890, 0.8902, 0.2321],
        [0.2672, 0.9081, 0.4632],
        [0.8986, 0.6640, 0.5581],
        [0.8408, 0.0572, 0.9438],
        [0.1794, 0.1074, 0.1593],
        [0.1144, 0.4986, 0.6153],
        [0.6553, 0.4977, 0.1595],
        [0.2953, 0.3734, 0.5736],
        [0.1230, 0.2536, 0.4071]])
tensor([1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1])


In [83]:
noised_atom_types, noised_coords, added_noise = permutate_al_atoms(coords, atom_types, noise)

Al indices tensor([ 5,  6, 10])
Row ind [ 0  9 17]
Col ind [1 0 2]


In [84]:
print(noised_atom_types)
print(added_noise)
print(noised_coords)

tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1])
tensor([[ 0.1354, -0.3066,  0.5192],
        [-0.1604, -0.1733, -0.3392],
        [-0.5753, -0.0448,  0.2315]])
tensor([[0.0165, 0.5861, 0.2502],
        [0.5998, 0.9031, 0.8464],
        [0.9129, 0.8250, 0.1391],
        [0.9210, 0.5640, 0.1489],
        [0.9313, 0.2000, 0.6341],
        [0.5801, 0.4492, 0.4428],
        [0.0977, 0.6570, 0.3139],
        [0.4812, 0.0150, 0.7531],
        [0.0948, 0.9301, 0.9043],
        [0.4719, 0.4274, 0.4080],
        [0.6529, 0.5688, 0.1706],
        [0.9890, 0.8902, 0.2321],
        [0.2672, 0.9081, 0.4632],
        [0.8986, 0.6640, 0.5581],
        [0.8408, 0.0572, 0.9438],
        [0.1794, 0.1074, 0.1593],
        [0.1144, 0.4986, 0.6153],
        [0.6553, 0.4977, 0.1595],
        [0.2953, 0.3734, 0.5736],
        [0.1230, 0.2536, 0.4071]])


In [8]:
original_al_indices = atom_types == 0
noised_al_indices = noised_atom_types == 0

In [9]:
print(original_al_indices)
print(noised_al_indices)


tensor([False, False, False, False, False,  True, False, False, False, False,
        False, False, False, False,  True, False,  True, False, False, False])
tensor([False,  True, False, False, False, False, False, False,  True, False,
        False,  True, False, False, False, False, False, False, False, False])


In [25]:
# Calculate manhatten distance between original and noise al coords
noised_distance = torch.sum(torch.abs(coords[original_al_indices] - noised_coords), dim=1)
#distance_matrix = torch.cdist(coords[original_al_indices], noised_coords)
print(noised_distance)
residual_distance = torch.sum(torch.abs(coords[noised_al_indices] - noised_coords), dim=1)
print(residual_distance)

tensor([1.1360, 0.9531, 1.3098])
tensor([0.8166, 1.1954, 0.7038])


In [23]:
print(coords[original_al_indices])
print(noised_coords)

tensor([[0.4780, 0.8582, 0.0117],
        [0.4945, 0.2263, 0.7024],
        [0.8900, 0.1269, 0.2140]])
tensor([[0.1750, 0.3822, 0.3687],
        [0.9580, 0.3030, 0.2895],
        [0.7026, 0.8182, 0.6451]])
